# Mission 2B: Rebuild Mary's Language Core

Epoch Zero NLP Bootcamp · Section 2: PyTorch and Text Classification with Neural Networks · Advanced track assignment

#### TRANSMISSION RECEIVED: SHIP HAIL MARY, DAY 231

Crew, Mary can now sort alarms from dinner menus, but she still cannot speak. This
morning Rocky tapped out a question and Mary answered: "the the hull cargo is... is."
Her language core is a ruin, and her reflexes are misfiring too.
"Mary... words come out wrong. Mary feel... gradient? Fix Mary. Please."

Your mission: re-derive Mary's learning reflexes, rebuild her language core two
different ways, tune her emergency filter, and read her black-box recorder before the
next jump.

#### Mission briefing
Estimated time: About 2 hours

Tools: Google Colab with NumPy, PyTorch and scikit-learn

Allowed help: Your Section 2 reading notes and the PyTorch and NumPy docs

Deliverable: One Colab notebook (.ipynb) with every cell run and all outputs visible

Deadline and submission: 03/11/26 noon 12pm via [google form link]

In [1]:
# Setup

import math, random
from collections import Counter, defaultdict
import numpy as np
import torch
import torch.nn as nn
seed = 42
torch.manual_seed(seed)
print('Seed:', seed)

Seed: 42


# Task A: Reflex arc

Mary's learning reflex is backpropagation. In topic 5 you derived it for a 2-layer network
with ReLU and softmax cross-entropy. Now implement it yourself and prove it is correct.

In [2]:
# Run this setup:

rng_a = np.random.default_rng(seed)
N, d, h, C = 8, 3, 5, 3
X = rng_a.normal(size=(N, d))
y = rng_a.integers(0, C, N)

params = {
    'W1': rng_a.normal(size=(d, h)) * 0.5, 'b1': np.zeros(h),
    'W2': rng_a.normal(size=(h, C)) * 0.5, 'b2': np.zeros(C),
}

def forward(params):
    Z1 = X @ params['W1'] + params['b1']
    A1 = np.maximum(0, Z1)
    Z2 = A1 @ params['W2'] + params['b2']
    Z2 = Z2 - Z2.max(axis=1, keepdims=True)
    P = np.exp(Z2)
    P /= P.sum(axis=1, keepdims=True)
    loss = -np.log(P[np.arange(N), y]).mean()
    return loss, (Z1, A1, P)

In [3]:
# (a) The backward pass. Fill in backward using the formulas in reading notes. It must
# return a gradient for every parameter, with the same shape as the parameter.

def backward(params, cache):
    Z1, A1, P = cache
    # TODO: dZ2, dW2, db2, dA1, dZ1, dW1, db1

    # Gradient of softmax cross-entropy
    dZ2 = P.copy()
    dZ2[np.arange(N), y] -= 1
    dZ2 /= N

    # Gradients of second layer
    dW2 = A1.T @ dZ2
    db2 = dZ2.sum(axis=0)

    # Backpropagate to hidden layer
    dA1 = dZ2 @ params['W2'].T

    # ReLU derivative
    dZ1 = dA1 * (Z1 > 0)

    # Gradients of first layer
    dW1 = X.T @ dZ1
    db1 = dZ1.sum(axis=0)
    
    return {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2}

In [4]:
# (b) The numerical judge. Fill in the centred-difference gradient from topic 5.4:

def numeric_grad(params, name, idx, eps=1e-5):
    # TODO: add eps to params[name][idx] and recompute the loss; subtract eps and recompute;
    # RESTORE the original value; return (loss_plus - loss_minus) / (2 * eps)

    # Store original parameter value
    original = params[name][idx]

    # Compute loss with +eps
    params[name][idx] = original + eps
    loss_plus, _ = forward(params)

    # Compute loss with -eps
    params[name][idx] = original - eps
    loss_minus, _ = forward(params)

    # Restore original value
    params[name][idx] = original

    # Centred-difference gradient
    return (loss_plus - loss_minus) / (2 * eps)

In [5]:
#(c) The verdict. Run this check. The relative error must be below 1e-6 for all four entries.
# (ReLU has a kink at 0, so if one entry is off by a lot, try a different index before suspecting your code.)

loss, cache = forward(params)

grads = backward(params, cache)

checks = [
    ('W1', (0, 0)), 
    ('W1', (d - 1, h - 1)), 
    ('W2', (2, 1)), 
    ('b2', (0,))
]

for name, idx in checks:
    a = grads[name][idx]
    n = numeric_grad(params, name, idx)

    rel = abs(a - n) / max(abs(a) + abs(n), 1e-12)
    
    print(name, idx, 'analytic', round(a, 6), 'numeric', round(n, 6), 'relerr', f'{rel:.2e}')

W1 (0, 0) analytic -0.001952 numeric -0.001952 relerr 2.74e-10
W1 (2, 4) analytic -0.028503 numeric -0.028503 relerr 4.20e-11
W2 (2, 1) analytic -0.148531 numeric -0.148531 relerr 2.28e-11
b2 (0,) analytic 0.044814 numeric 0.044814 relerr 1.01e-11


In [6]:
#(d) Let Mary learn. Run 100 steps of gradient descent (learning rate 0.5) with your
#gradients. Print the loss at steps 0, 50 and 99, next to math.log(C) .

learning_rate = 0.5

print("(d) Gradient Descent Training:\n")

for step in range(100):

    # Forward pass
    loss, cache = forward(params)

    # Backward pass
    grads = backward(params, cache)

    # Print loss at required steps
    if step in [0, 50, 99]:
        print(
            f"Step {step}: "
            f"Loss = {loss:.6f}, "
            f"math.log(C) = {math.log(C):.6f}"
        )

    # Update parameters
    for name in params:
        params[name] -= learning_rate * grads[name]

(d) Gradient Descent Training:

Step 0: Loss = 1.048768, math.log(C) = 1.098612
Step 50: Loss = 0.100760, math.log(C) = 1.098612
Step 99: Loss = 0.041384, math.log(C) = 1.098612


# Task B: Rebuild the language core

Mary needs to predict the next word of her own ship log. You will build two language
models, a counting model and a neural model, and judge them with perplexity.

Run this cell to generate Mary’s ship log:

In [7]:
rng = random.Random(seed)

sections = [
    'engine bay', 'cargo hold', 'lab', 'bridge', 
    'airlock', 'reactor room'
]

findings = [
    'astrophage', 'a cracked pipe', 'loose cables', 
    'a warm panel', 'nothing unusual'
]

templates = [
    'mary scans the {sec} and finds {obj} .',
    'grace checks the {sec} for {obj} .',
    'rocky hums near the {sec} and finds {obj} .',
    'the crew repairs the {sec} after {obj} .'
]

log = [rng.choice(templates).format(sec=rng.choice(sections), obj=rng.choice(findings)) for _ in range(400)]
tokens = ' '.join(log).split()
split = int(len(tokens) * 0.8)
train_tokens, test_tokens = tokens[:split], tokens[split:]
vocab = sorted(set(tokens))
stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for w, i in stoi.items()}

V = len(vocab)
K = 3 # context length for the neural model

print(V, len(train_tokens), len(test_tokens))
print(' '.join(tokens[:12]))

34 3128 782
mary scans the engine bay and finds loose cables . grace checks


B1: The counting core. Build a bigram model from train_tokens with add-one
smoothing: P(next | prev) = (count(prev, next) + 1) / (count(prev) + V) .
Compute the perplexity on the test tokens from index K onward (so both models are
scored on the same words). Print it.

In [9]:
from collections import Counter

# Count bigrams and previous-word occurrences
bigram_counts = Counter(
    zip(train_tokens[:-1], train_tokens[1:])
)

prev_counts = Counter(train_tokens[:-1])

# Calculate test perplexity from index K onward
log_probs = []

for i in range(K, len(test_tokens)):

    prev_word = test_tokens[i - 1]
    next_word = test_tokens[i]

    probability = (
        bigram_counts[(prev_word, next_word)] + 1
    ) / (
        prev_counts[prev_word] + V
    )

    log_probs.append(math.log(probability))

bigram_test_ppl = math.exp(-sum(log_probs) / len(log_probs))

print("B1: Bigram Test Perplexity:", round(bigram_test_ppl, 4))

B1: Bigram Test Perplexity: 2.5304


B2: The neural core. Write an nn.Module named MaryLM like topic 7.4: embed the
previous K = 3 words (embedding size 16), concatenate, Linear(K*16, 64) , tanh ,
Linear(64, V) . Train it full-batch with CrossEntropyLoss and Adam ( lr=0.01 ) for 200
epochs on windows built from train_tokens . Print the train and test perplexity ( exp of
the loss) at epochs 50, 100, 150 and 200. Test windows must predict exactly
test_tokens[K:] .

In [10]:
#Prepare Training and Testing Windows

# Convert tokens into numerical IDs
train_ids = [stoi[w] for w in train_tokens]
test_ids = [stoi[w] for w in test_tokens]


def make_windows(token_ids, K=3):

    X = []
    y = []

    for i in range(K, len(token_ids)):

        X.append(token_ids[i-K:i])
        y.append(token_ids[i])

    return (
        torch.tensor(X, dtype=torch.long),
        torch.tensor(y, dtype=torch.long)
    )


X_train, y_train = make_windows(train_ids, K)
X_test, y_test = make_windows(test_ids, K)

print("Training input shape:", X_train.shape)
print("Training target shape:", y_train.shape)

print("Testing input shape:", X_test.shape)
print("Testing target shape:", y_test.shape)

# Verify test targets correspond exactly to test_tokens[K:]
assert y_test.tolist() == test_ids[K:]

print("Test target verification: Passed")

Training input shape: torch.Size([3125, 3])
Training target shape: torch.Size([3125])
Testing input shape: torch.Size([779, 3])
Testing target shape: torch.Size([779])
Test target verification: Passed


In [11]:
# Define MaryLM
class MaryLM(nn.Module):

    def __init__(self, vocab_size, K=3):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, 16)

        self.fc1 = nn.Linear(K * 16, 64)
        self.tanh = nn.Tanh()

        self.fc2 = nn.Linear(64, vocab_size)

    def forward(self, x):

        # x shape: (batch_size, K)
        embedded = self.embedding(x)

        # Concatenate embeddings
        flattened = embedded.reshape(x.size(0), -1)

        # Hidden layer
        hidden = self.fc1(flattened)
        hidden = self.tanh(hidden)

        # Output raw logits
        logits = self.fc2(hidden)

        return logits


# Initialize model
torch.manual_seed(seed)

lm_model = MaryLM(V, K)

print(lm_model)

MaryLM(
  (embedding): Embedding(34, 16)
  (fc1): Linear(in_features=48, out_features=64, bias=True)
  (tanh): Tanh()
  (fc2): Linear(in_features=64, out_features=34, bias=True)
)


In [12]:
# Train for 200 Epochs

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    lm_model.parameters(),
    lr=0.01
)

epochs = 200

print("\nB2: Neural Language Model Training\n")

for epoch in range(1, epochs + 1):

    lm_model.train()

    # Forward pass
    train_logits = lm_model(X_train)

    # Compute loss
    loss = criterion(train_logits, y_train)

    # Backpropagation
    optimizer.zero_grad()
    loss.backward()

    # Update parameters
    optimizer.step()

    # Evaluate every 50 epochs
    if epoch % 50 == 0:

        lm_model.eval()

        with torch.no_grad():

            train_loss = criterion(
                lm_model(X_train), y_train
            )

            test_loss = criterion(
                lm_model(X_test), y_test
            )

            train_ppl = math.exp(train_loss.item())
            test_ppl = math.exp(test_loss.item())

        print(
            f"Epoch {epoch:3d} | "
            f"Train PPL: {train_ppl:.4f} | "
            f"Test PPL: {test_ppl:.4f}"
        )


B2: Neural Language Model Training

Epoch  50 | Train PPL: 1.7041 | Test PPL: 1.7166
Epoch 100 | Train PPL: 1.6960 | Test PPL: 1.7215
Epoch 150 | Train PPL: 1.6942 | Test PPL: 1.7246
Epoch 200 | Train PPL: 1.6934 | Test PPL: 1.7263


B3: The verdict. Print a two-row table of the test perplexity of both models. In 2 to 3
sentences, say which model wins and why, using ideas from topic 7 (context length,
embeddings, smoothing).

In [13]:
# Calculate final neural model perplexity
lm_model.eval()

with torch.no_grad():
    final_test_loss = criterion(
        lm_model(X_test), y_test
    )

neural_test_ppl = math.exp(final_test_loss.item())

print("\nB3: Test Perplexity Comparison")
print("-" * 40)
print(f"{'Model':<20} {'Test Perplexity':>18}")
print("-" * 40)

print(f"{'Bigram Model':<20} {bigram_test_ppl:>18.4f}")
print(f"{'Neural Model':<20} {neural_test_ppl:>18.4f}")

print("-" * 40)

if neural_test_ppl < bigram_test_ppl:
    print("Winner: Neural Model")
elif bigram_test_ppl < neural_test_ppl:
    print("Winner: Bigram Model")
else:
    print("Result: Tie")


B3: Test Perplexity Comparison
----------------------------------------
Model                   Test Perplexity
----------------------------------------
Bigram Model                     2.5304
Neural Model                     1.7263
----------------------------------------
Winner: Neural Model


The neural language model performs better, achieving a test perplexity of 1.7263, compared to 2.5304 for the bigram model. This is because the neural model uses the previous three words as context and learns word embeddings to capture patterns in the text, whereas the bigram model considers only the previous word. Although add-one smoothing helps the bigram model handle unseen word pairs, the neural model predicts the next word more accurately on this dataset.

B4: Turn up the heat.

(a) For the context ['bay', 'and', 'finds'] , print the top 3 most probable next
words with their probabilities at temperature 1.0 and at temperature 0.5.

(b) Starting from ['mary', 'scans', 'the'] , generate 12 more words at
temperature 0.5 and at temperature 1.5 (call torch.manual_seed(seed) before each
run). Print both.

(c) In 2 sentences, describe what temperature did to the probabilities in (a) and to the
text in (b).

In [14]:
# (a): Top 3 next-word predictions

context = ['bay', 'and', 'finds']

context_ids = torch.tensor(
    [[stoi[w] for w in context]],
    dtype=torch.long
)

lm_model.eval()

with torch.no_grad():
    logits = lm_model(context_ids)

    for temperature in [1.0, 0.5]:

        probs = torch.softmax(
            logits / temperature,
            dim=-1
        )

        top_probs, top_indices = torch.topk(
            probs[0], k=3
        )

        print(f"\nTemperature: {temperature}")

        for idx, prob in zip(top_indices, top_probs):
            word = itos[idx.item()]
            print(f"{word}: {prob.item():.4f}")


Temperature: 1.0
a: 0.5327
nothing: 0.2337
astrophage: 0.2007

Temperature: 0.5
a: 0.7472
nothing: 0.1439
astrophage: 0.1061


In [15]:
# (b): Generate 12 Words at Two Temperatures
def generate_text(model, start_words, vocab, itos,
                  temperature=1.0, num_words=12):

    model.eval()

    words = start_words.copy()

    for _ in range(num_words):

        # Take the previous K words
        context = words[-K:]

        # Convert words into IDs
        context_ids = torch.tensor(
            [[vocab[w] for w in context]],
            dtype=torch.long
        )

        with torch.no_grad():

            # Predict next-word logits
            logits = model(context_ids)

            # Apply temperature
            probs = torch.softmax(
                logits / temperature,
                dim=-1
            )

            # Sample next word
            next_id = torch.multinomial(
                probs[0], num_samples=1
            ).item()

        # Convert ID to word
        next_word = itos[next_id]

        # Append generated word
        words.append(next_word)

    return ' '.join(words)


start_words = ['mary', 'scans', 'the']

# Temperature 0.5
torch.manual_seed(seed)

text_low = generate_text(
    lm_model,
    start_words,
    stoi,
    itos,
    temperature=0.5,
    num_words=12
)

print("Temperature 0.5:")
print(text_low)


# Temperature 1.5
torch.manual_seed(seed)

text_high = generate_text(
    lm_model,
    start_words,
    stoi,
    itos,
    temperature=1.5,
    num_words=12
)

print("\nTemperature 1.5:")
print(text_high)

Temperature 0.5:
mary scans the engine bay for a cracked pipe . the crew repairs the bridge

Temperature 1.5:
mary scans the engine bay for a cracked pipe . the crew repairs the bridge


#### (c) : Reflection — Effect of Temperature

At temperature 0.5, the probability distribution becomes sharper, making the most likely next words more probable than at temperature 1.0. In text generation, the lower temperature generally produces more predictable sentences, while temperature 1.5 allows more varied and sometimes less coherent word choices.

# Task C: The emergency filter

Mary must flag real DISTRESS calls in a stream of 1,000 messages: 950 routine and 50
distress. Two candidate filters were tested. Counts below treat distress as the positive
class:

In [17]:
model_a = dict(TP=10, FN=40, FP=5, TN=945)
model_b = dict(TP=40, FN=10, FP=60, TN=890)

(a) Write report(m) that returns the accuracy, the precision, recall and F1 of the
distress class, the F1 of the routine class (hint: for the routine class, TN plays the role
of TP, FN becomes FP and FP becomes FN) and the macro-F1 (the average of the two
class F1 scores). Print everything for both models.

(b) Print the class weights for counts [950, 50] using total / (num_classes *
count) , and write the one PyTorch line that gives them to the loss function (as a
comment or code).

(c) Which filter should Mary deploy on a ship where missing a distress call can kill the
crew? Answer in 2 to 3 sentences, using your numbers, and say why accuracy alone
would have misled you.

In [18]:
# (a) Calculate Evaluation Metrics

model_a = dict(TP=10, FN=40, FP=5, TN=945)
model_b = dict(TP=40, FN=10, FP=60, TN=890)

def report(m):
    TP, FN, FP, TN = m['TP'], m['FN'], m['FP'], m['TN']

    # Accuracy
    accuracy = (TP + TN) / (TP + TN + FP + FN)

    # Distress class metrics
    precision = TP / (TP + FP)
    recall = TP / (TP + FN)
    f1_distress = 2 * precision * recall / (precision + recall)

    # Routine class metrics
    precision_routine = TN / (TN + FN)
    recall_routine = TN / (TN + FP)
    f1_routine = 2 * precision_routine * recall_routine / (precision_routine + recall_routine)

    # Macro F1
    macro_f1 = (f1_distress + f1_routine) / 2

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_distress': f1_distress,
        'f1_routine': f1_routine,
        'macro_f1': macro_f1
    }


# Print results
for name, model in [('Model A', model_a), ('Model B', model_b)]:
    results = report(model)

    print(f"\n{name}")
    for metric, value in results.items():
        print(f"{metric}: {value:.4f}")


Model A
accuracy: 0.9550
precision: 0.6667
recall: 0.2000
f1_distress: 0.3077
f1_routine: 0.9767
macro_f1: 0.6422

Model B
accuracy: 0.9300
precision: 0.4000
recall: 0.8000
f1_distress: 0.5333
f1_routine: 0.9622
macro_f1: 0.7477


In [19]:
# (b) Calculate Class Weights

import torch
import torch.nn as nn

counts = [950, 50]

total = sum(counts)
num_classes = len(counts)

class_weights = [
    total / (num_classes * count)
    for count in counts
]

print("Class Weights:", class_weights)

# PyTorch weighted cross-entropy loss
criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(class_weights, dtype=torch.float32)
)

Class Weights: [0.5263157894736842, 10.0]


#### (c) Which Filter Should Mary Deploy?

Mary should choose Model B because it detects 80% of distress calls (40 out of 50), while Model A detects only 20% (10 out of 50). Although Model A has higher accuracy (95.5% versus 93%), it misses 40 distress calls compared to only 10 missed by Model B, which could be dangerous for the crew. Accuracy alone is misleading because most messages are routine, so recall and macro-F1 are more useful for evaluating the emergency filter.

# Task D: The black-box recorder

Mary's flight recorder logged 12 training epochs of an old model. Generate the log:

In [21]:
best = 6

# hidden from you: find it from the numbers!
epochs = np.arange(1, 13)

train_loss = 0.7 * np.exp(-0.28 * epochs)
val_loss = 0.7 * np.exp(-0.28 * np.minimum(epochs, best)) + 0.03 + 0.02 * np.maximum(0, epochs - best) ** 1.5

(a) Print a table with the columns epoch, train loss and validation loss (3 decimals).

(b) Using code (for example np.argmin ), report the epoch with the lowest validation
loss.

(c) Print the gap (validation minus train) at epoch 12.

(d) Diagnose the problem in one sentence, and name two fixes from topic 6.6.

(e) If you had used early stopping with patience=2 , at which epoch would training
have stopped, and which epoch's weights would you have restored? (Use the logic of
the code in topic 6.6.)

In [22]:
# (a) Print loss table
print("(a) Training and Validation Losses")
print("-" * 40)
print(f"{'Epoch':<10}{'Train Loss':<15}{'Val Loss':<15}")
print("-" * 40)

for epoch, train, val in zip(epochs, train_loss, val_loss):
    print(f"{epoch:<10}{train:<15.3f}{val:<15.3f}")

(a) Training and Validation Losses
----------------------------------------
Epoch     Train Loss     Val Loss       
----------------------------------------
1         0.529          0.559          
2         0.400          0.430          
3         0.302          0.332          
4         0.228          0.258          
5         0.173          0.203          
6         0.130          0.160          
7         0.099          0.180          
8         0.075          0.217          
9         0.056          0.264          
10        0.043          0.320          
11        0.032          0.384          
12        0.024          0.454          


In [24]:
# (b) Find epoch with minimum validation loss
best_epoch = epochs[np.argmin(val_loss)]
print("\n(b) Best Epoch:", best_epoch)
print(f"Minimum Validation Loss: {np.min(val_loss):.4f}")


(b) Best Epoch: 6
Minimum Validation Loss: 0.1605


In [25]:
# (c) Validation-training gap at epoch 12
gap = val_loss[-1] - train_loss[-1]

print("\n(c) Gap at Epoch 12:")
print(f"Validation Loss - Training Loss = {gap:.4f}")


(c) Gap at Epoch 12:
Validation Loss - Training Loss = 0.4301


#### (d) Diagnose the Problem and Suggest Two Fixes

The model is overfitting because its training loss keeps decreasing while its validation loss increases after epoch 6; two possible fixes are early stopping and stronger regularization, such as dropout or weight decay.

In [26]:
# (e) Early Stopping With Patience = 2

patience = 2
best_val_loss = float('inf')
best_epoch = 0
counter = 0

for epoch, loss in zip(epochs, val_loss):

    if loss < best_val_loss:
        best_val_loss = loss
        best_epoch = epoch
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print("Training stopped at epoch:", epoch)
        print("Restore weights from epoch:", best_epoch)
        break

Training stopped at epoch: 8
Restore weights from epoch: 6


#### Submission checklist

Notebook runs top to bottom (Runtime → Run all) with all outputs visible

Your own crew code is in the first cell

Every written answer (verdicts, diagnoses) is in a text cell, in your own words

Notebook shared with link access

"Mary... speak. Mary speak! Thank you, crew. Mary hear Rocky now."
🚀